# Reversing BERT Embeddings

Given the *output* of a BERT submodule, recover the *inputs* that could have produced it.

Ported from `FantasyArchetypesInSemanticSpace/notebooks/SimilarityBetweenHiddenLayers.ipynb`,
keeping only the **closed-form / analytic** inversions. Gradient-descent search, brute-force
token sweeps, layer-similarity measurements and the geometry plots were deliberately left behind.

Model: `BAAI/bge-large-en-v1.5` — 24-layer BERT, hidden dim 1024, 16 heads x 64, FFN dim 4096.

## 1. Setup: run the model, get embeddings

In [1]:
import torch
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model_name = "BAAI/bge-large-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

HIDDEN_DIM = model.config.hidden_size          # 1024
FFN_DIM = model.config.intermediate_size       # 4096
NUM_HEADS = model.config.num_attention_heads   # 16
HEAD_DIM = HIDDEN_DIM // NUM_HEADS             # 64
NUM_LAYERS = model.config.num_hidden_layers    # 24
print(f'hidden={HIDDEN_DIM} ffn={FFN_DIM} heads={NUM_HEADS} head_dim={HEAD_DIM} layers={NUM_LAYERS}')

/home/dmmsp/Projects/ReversingEmbeddings/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


hidden=1024 ffn=4096 heads=16 head_dim=64 layers=24


In [2]:
def cosine_similarity(embedding1, embedding2):
    embedding1_normalized = F.normalize(embedding1, p=2, dim=0)
    embedding2_normalized = F.normalize(embedding2, p=2, dim=0)
    return torch.dot(embedding1_normalized, embedding2_normalized).item()


def getEmbedding(text):
    """Sentence embedding (pooler output) for a phrase."""
    outputs = model(**tokenizer(text, return_tensors='pt').to(device))
    return outputs.pooler_output[0]


def getNumpyEmbedding(text):
    return getEmbedding(text).cpu().detach().clone().numpy()

In [3]:
targetText = 'magic is real'

encoded = tokenizer(targetText, return_tensors='pt').to(device)
print(encoded)

outputs = model(**encoded, output_hidden_states=True)
target_embedding = outputs.pooler_output[0]

print(f'\npooler_output: {target_embedding.shape}')
print(target_embedding)
print(f'\nhidden_states: {len(outputs.hidden_states)} tensors of {outputs.hidden_states[0].shape}')

{'input_ids': tensor([[ 101, 3894, 2003, 2613,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}



pooler_output: torch.Size([1024])
tensor([-0.9450, -0.8097, -0.8362,  ...,  0.4859,  0.9817, -0.8716],
       grad_fn=<SelectBackward0>)

hidden_states: 25 tensors of torch.Size([1, 5, 1024])


In [4]:
SEQ_LEN = encoded['input_ids'].shape[1]
print(f'sequence length: {SEQ_LEN} tokens -> {tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])}')

sequence length: 5 tokens -> ['[CLS]', 'magic', 'is', 'real', '[SEP]']


### Capturing what a submodule actually saw

To reverse a submodule we need its true input/output pair. A forward hook grabs both.

In [5]:
from contextlib import contextmanager

@contextmanager
def capture(module):
    """Capture input/output tensors of a submodule during a forward pass.

    Usage:
        with capture(model.encoder.layer[23].output.dense) as cap:
            model.encoder.layer[23](outputs.hidden_states[23])
        cap['input'], cap['output']
    """
    store = {}

    def hook_function(mod, inp, out):
        store['input'] = inp[0]
        store['output'] = out

    handle = module.register_forward_hook(hook_function)
    try:
        yield store
    finally:
        handle.remove()


# The layer we will pull apart. Feeding it hidden_states[23] reproduces hidden_states[24].
LAYER = 23
layer = model.encoder.layer[LAYER]
layer_input = outputs.hidden_states[LAYER]

def np_(t):
    return t.cpu().detach().clone().numpy()

## 2. The reversal primitives

Each one answers: *given this output, what inputs produce it?*

### 2.1 Linear layer: `y = Wx`

`W` is generally not square, so there is no plain inverse. The pseudoinverse gives one
particular solution, and `(I - W⁺W)z` spans the nullspace — every input that produces
the same output. Pick any `z` and you get another valid answer.

In [6]:
def allInputsForMatrixProduct(y, W):
    W_pinv = np.linalg.pinv(W)

    # Create nullspace projection matrix
    I = np.eye(W.shape[1])
    nullspace_proj = I - W_pinv @ W

    # Return function that generates solutions for any z
    def solution_generator(z):
        particular_solution = W_pinv @ y
        nullspace_component = nullspace_proj @ z
        print("Nullspace component magnitude:", np.linalg.norm(nullspace_proj @ z))
        print("Particular solution magnitude:", np.linalg.norm(particular_solution))
        return particular_solution + nullspace_component

    return solution_generator

### 2.2 Picking the `z` that recovers the *original* input

The family above contains the true input somewhere. Solve `(I - W⁺W)z = x - W⁺y` for the
`z` that lands exactly on it.

In [7]:
def find_z_for_exact_match(original_input, target_output, weights):
    # Get pseudoinverse
    W_pinv = np.linalg.pinv(weights)

    # Calculate nullspace projection matrix
    I = np.eye(weights.shape[1])
    nullspace_proj = I - W_pinv @ weights

    # Solve for z: (I - W+W)z = x - W+y
    # Using pseudoinverse again because (I - W+W) might be singular
    left_side = original_input - (W_pinv @ target_output)
    z = np.linalg.pinv(nullspace_proj) @ left_side

    # Verify the solution
    reconstructed = W_pinv @ target_output + nullspace_proj @ z
    print("Reconstruction error:", np.linalg.norm(reconstructed - original_input))

    return z

### 2.3 Dense + tanh: `y = tanh(Wx + b)` — the pooler

`arctanh` undoes the squash, then it reduces to the linear case.

In [8]:
def find_all_possible_inputs(y, W, b):
    """
    Find all possible inputs x that satisfy y = tanh(Wx + b)

    Parameters:
    y: target output vector (must have all elements between -1 and 1)
    W: weight matrix
    b: bias vector

    Returns:
    function that generates solutions based on input z
    """
    # Check if y is valid (all elements between -1 and 1)
    if not np.all(np.abs(y) < 1):
        raise ValueError("All elements of y must be between -1 and 1")

    # Step 1: Apply arctanh
    arctanh_y = np.arctanh(y)

    # Step 2: Subtract bias
    target = arctanh_y - b

    # Step 3: Find pseudoinverse of W
    W_pinv = np.linalg.pinv(W)

    # Create nullspace projection matrix
    I = np.eye(W.shape[1])
    nullspace_proj = I - W_pinv @ W

    # Return function that generates solutions for any z
    def solution_generator(z):
        particular_solution = W_pinv @ target
        nullspace_component = nullspace_proj @ z
        print("Nullspace component magnitude:", np.linalg.norm(nullspace_proj @ z))
        print("Particular solution magnitude:", np.linalg.norm(particular_solution))
        return particular_solution + nullspace_component

    return solution_generator

### 2.4 LayerNorm — reversible except for two numbers

Rescales each token's vector to mean 0 / variance 1, then applies a learned scale and shift.

The learned affine undoes cleanly. The normalization does not — but it costs less than it
looks like. A normalized vector has mean 0 and std 1 by construction, so the *only* thing that
cannot be recovered is the input's original mean and standard deviation: **2 numbers per token
out of 1024**. The direction is fully preserved. `input_scale` and `input_shift` below are those
two free parameters; feed the true ones back in and the input returns exactly.

In [9]:
def generate_possible_inputs(normalized_values, weight, bias, input_scale, input_shift):
    # First undo LayerNorm's learned transformations
    unlearned = [(x - bias[idx]) / weight[idx] for idx, x in enumerate(normalized_values)]
    # Now apply our input parameters to generate a possible input sequence
    # Any sequence generated this way would normalize to our target output
    return [x * input_scale + input_shift for x in unlearned]

### 2.5 GELU — Newton's method

GELU has no closed-form inverse, but it is monotonic over the relevant range, so Newton
iteration converges deterministically. This is a numerical root-find, not a search.

In [10]:
from scipy.special import erf

def gelu_vectorized(x):
    """Vectorized GELU activation"""
    return 0.5 * x * (1 + erf(x / np.sqrt(2)))

def gelu_derivative_vectorized(x):
    """Vectorized GELU derivative"""
    return 0.5 * (1 + erf(x / np.sqrt(2))) + \
           (x * np.exp(-(x**2) / 2)) / (2 * np.sqrt(2 * np.pi))

def inverse_gelu_matrix(target_matrix, max_iter=50, tol=1e-7):
    """
    Vectorized Newton's method for GELU inverse on matrices

    Args:
        target_matrix: Matrix of target values
        max_iter: Maximum iterations
        tol: Convergence tolerance
    """
    # Initial guess
    x = target_matrix * 1.7  # Vectorized initial guess

    for _ in range(max_iter):
        # Compute function value and derivative
        fx = gelu_vectorized(x) - target_matrix
        fpx = gelu_derivative_vectorized(x)

        # Newton step
        step = fx / fpx
        x_new = x - step

        # Check convergence
        if np.max(np.abs(step)) < tol:
            return x_new

        x = x_new

    return x  # Return best estimate if max_iter reached

### 2.6 Softmax — reversible up to an additive constant

`softmax(x) = softmax(x + c)`, so `log(y)` plus any constant is a valid input. A whole
one-dimensional family per row.

In [11]:
def get_softmax_input_space(y, num_samples=5):
    """
    Given a softmax output y, returns a set of possible input matrices that would produce y.

    Args:
        y (np.ndarray): A matrix of softmax outputs (each row sums to 1)
        num_samples (int): Number of sample solutions to return

    Returns:
        np.ndarray: Shape (num_samples, *y.shape) array where each element along axis 0
                   is a possible input that would produce y through softmax
    """
    # Verify input is valid softmax output
    if not np.allclose(np.sum(y, axis=-1), 1.0):
        raise ValueError("Input must be valid softmax output (rows must sum to 1)")

    # Take log of the outputs
    base_solution = np.log(y)

    # Generate random constants to demonstrate the solution space
    # Using both positive and negative constants centered around 0
    constants = np.linspace(-10, 10, num_samples)

    # Create multiple solutions by adding different constants
    solutions = np.zeros((num_samples, *y.shape))
    for i, c in enumerate(constants):
        solutions[i] = base_solution + c

    return solutions


def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def verify_solutions(solutions, original_output):
    """
    Verify that all solutions produce the original output when passed through softmax.
    """
    for solution in solutions:
        if not np.allclose(softmax(solution), original_output):
            return False
    return True

### 2.7 The attention value path

With the attention probabilities held fixed, `probs @ (X @ Wv.T + bv) = out` is linear in `X`,
so two pseudoinverses recover it.

In [12]:
def solve_for_attention_input(attention_probs, value_weight, value_bias, calculated_output,
                              num_heads=16, head_dim=64, batch_size=1, seq_len=5):
    """
    Solves for attentionInput in the equation:
    attention_probs @ (input @ value_weight.T + value_bias) = calculated_output
    taking into account the shape transformations in multi-head attention
    """
    hidden_dim = num_heads * head_dim

    # First reshape calculated_output back to attention space
    # [batch, seq_len, hidden_dim] -> [batch, num_heads, seq_len, head_dim]
    output_attention_space = calculated_output.view(batch_size, seq_len, num_heads, head_dim)
    output_attention_space = output_attention_space.transpose(1, 2)

    # attention_probs shape: [batch, num_heads, seq_len, seq_len]
    # Solve for (input @ value_weight.T + value_bias) using pseudoinverse
    attention_probs_pinv = torch.linalg.pinv(attention_probs)  # Use pinv for each head
    intermediate = torch.matmul(attention_probs_pinv, output_attention_space)
    # intermediate shape: [batch, num_heads, seq_len, head_dim]

    # Reshape intermediate back to [batch, seq_len, hidden_dim]
    intermediate = intermediate.transpose(1, 2).contiguous()
    intermediate = intermediate.view(batch_size, seq_len, hidden_dim)

    # Solve for input: intermediate = input @ value_weight.T + value_bias
    intermediate = intermediate - value_bias

    # Solve for input using pseudoinverse of value_weight
    value_weight_pinv = torch.linalg.pinv(value_weight.T)
    input_matrix = intermediate @ value_weight_pinv

    return input_matrix


def verify_solution(input_matrix, attention_probs, value_weight, value_bias, calculated_output,
                    num_heads=16, head_dim=64, batch_size=1, seq_len=5):
    """Verifies if the solved input produces the correct output"""
    values = input_matrix @ value_weight.T + value_bias
    values = values.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)

    output = torch.matmul(attention_probs, values)
    output = output.transpose(1, 2).contiguous()
    output = output.view(batch_size, seq_len, num_heads * head_dim)

    error = torch.norm(output - calculated_output)
    return error.item()

### 2.8 Is a matrix invertible at all?

`checkNullspace` counts singular values below `1e-10`. **That tolerance is far too strict** —
the weights are float32, whose precision is ~1e-7, so a singular value of 1e-6 is
indistinguishable from zero in practice even though this function calls it nonzero.

`numericalRank` below uses the standard tolerance, `sigma_max * max(m,n) * eps`. The two
disagree substantially, and the second one is right. Section 5 shows the damage.

In [13]:
from numpy.linalg import svd

def checkNullspace(A):
    # Compute singular values
    _, s, _ = svd(A)

    # Count near-zero singular values (these correspond to nullspace dimensions)
    # Using a tolerance since floating point numbers might not be exactly zero
    tol = 1e-10  # You may need to adjust this tolerance
    nullspace_dim = sum(abs(s) < tol)

    return nullspace_dim

In [14]:
EPS32 = np.finfo(np.float32).eps   # 1.19e-07

def numericalRank(A, eps=EPS32):
    """Nullspace dimension at the tolerance float32 precision actually justifies."""
    s = np.linalg.svd(A, compute_uv=False)
    tol = s[0] * max(A.shape) * eps
    return int((s < tol).sum()), s[0] / s[min(A.shape) - 1]   # (dims lost, condition number)

## 3. Walking layer 23 backwards

Take the last encoder layer apart in reverse order:

`output.LayerNorm` <- `output.dense` <- `GELU` <- `intermediate.dense` <- `attention.output.dense` <- `self-attention`

### 3.1 `output.dense` — the FFN projection back down (4096 -> 1024)

In [15]:
with capture(layer.output.dense) as cap:
    layer(layer_input)
outputDenseInput, outputDenseOutput = cap['input'], cap['output']

print('Dense Input:', outputDenseInput.shape)
print(outputDenseInput[0])
print('Dense Output:', outputDenseOutput.shape)
print(outputDenseOutput[0])

# Strip the bias so we are left with a pure matrix product y = Wx
outputDenseOutputNoBias = np_(outputDenseOutput[0] - layer.output.dense.bias)
print('\nWithout bias:')
print(outputDenseOutputNoBias)

Dense Input: torch.Size([1, 5, 4096])
tensor([[-1.7454e-05, -5.1538e-02, -6.6088e-05,  ..., -9.9013e-05,
         -1.7445e-02, -1.4302e-04],
        [-1.8225e-04, -5.9708e-02, -3.3347e-04,  ..., -1.1227e-03,
         -2.0496e-02, -7.1462e-05],
        [-5.3588e-05, -6.5705e-02, -2.5427e-04,  ..., -7.2981e-05,
         -2.7109e-02, -1.1592e-04],
        [-1.8564e-04, -8.7957e-02, -9.1989e-04,  ..., -7.3797e-05,
         -4.2538e-02, -5.4416e-05],
        [-1.1813e-05, -1.1158e-01, -1.8291e-04,  ..., -1.1700e-03,
         -4.1407e-02, -1.9369e-04]], grad_fn=<SelectBackward0>)
Dense Output: torch.Size([1, 5, 1024])
tensor([[-0.0252, -0.1483, -0.0521,  ..., -0.0213, -0.0147,  0.0333],
        [-0.1382, -0.1020, -0.1916,  ...,  0.0017, -0.0260, -0.1250],
        [-0.0498, -0.1138, -0.0421,  ..., -0.0419, -0.0629,  0.0403],
        [-0.0752, -0.1497, -0.1350,  ..., -0.0377, -0.0197,  0.0146],
        [-0.1024, -0.1323, -0.1386,  ..., -0.1830,  0.0392, -0.0618]],
       grad_fn=<SelectBackwar

In [16]:
from tqdm import tqdm

outputDenseWeight = np_(layer.output.dense.weight)
print(f'weight shape: {outputDenseWeight.shape}  (1024 out x 4096 in -> 3072-dim nullspace)')

solutionGens = []
for i in tqdm(outputDenseOutputNoBias, desc='building solution families'):
    solutionGens.append(allInputsForMatrixProduct(i, outputDenseWeight))

weight shape: (1024, 4096)  (1024 out x 4096 in -> 3072-dim nullspace)


building solution families:   0%|          | 0/5 [00:00<?, ?it/s]

building solution families:  20%|██        | 1/5 [00:25<01:41, 25.38s/it]

building solution families:  40%|████      | 2/5 [01:04<01:40, 33.41s/it]

building solution families:  60%|██████    | 3/5 [01:16<00:47, 23.52s/it]

building solution families:  80%|████████  | 4/5 [01:25<00:17, 17.81s/it]

building solution families: 100%|██████████| 5/5 [01:39<00:00, 16.63s/it]

building solution families: 100%|██████████| 5/5 [01:39<00:00, 19.95s/it]

In [17]:
# z = 0 picks the minimum-norm member of each solution family
dims = np.zeros(FFN_DIM)
resultingDenseInput = [gen(dims) for gen in solutionGens]

print('\nOur reconstructed input:')
print(torch.tensor(np.array(resultingDenseInput)))
print('\nThe actual input:')
print(outputDenseInput[0])

Nullspace component magnitude: 0.0
Particular solution magnitude: 1.3765213
Nullspace component magnitude: 0.0
Particular solution magnitude: 1.8995526
Nullspace component magnitude: 0.0
Particular solution magnitude: 1.7386352
Nullspace component magnitude: 0.0
Particular solution magnitude: 1.9867663


Nullspace component magnitude: 0.0
Particular solution magnitude: 2.301452

Our reconstructed input:


tensor([[ 0.0133, -0.0485, -0.0237,  ...,  0.0082, -0.0318,  0.0128],
        [-0.0079, -0.0580, -0.0351,  ..., -0.0158, -0.0355,  0.0397],
        [-0.0003, -0.0606, -0.0430,  ...,  0.0091, -0.0423,  0.0330],
        [-0.0084, -0.0728, -0.0383,  ..., -0.0002, -0.0533,  0.0137],
        [-0.0059, -0.0785, -0.0546,  ..., -0.0221, -0.0265,  0.0218]],
       dtype=torch.float64)

The actual input:
tensor([[-1.7454e-05, -5.1538e-02, -6.6088e-05,  ..., -9.9013e-05,
         -1.7445e-02, -1.4302e-04],
        [-1.8225e-04, -5.9708e-02, -3.3347e-04,  ..., -1.1227e-03,
         -2.0496e-02, -7.1462e-05],
        [-5.3588e-05, -6.5705e-02, -2.5427e-04,  ..., -7.2981e-05,
         -2.7109e-02, -1.1592e-04],
        [-1.8564e-04, -8.7957e-02, -9.1989e-04,  ..., -7.3797e-05,
         -4.2538e-02, -5.4416e-05],
        [-1.1813e-05, -1.1158e-01, -1.8291e-04,  ..., -1.1700e-03,
         -4.1407e-02, -1.9369e-04]], grad_fn=<SelectBackward0>)


The reconstruction is **not** the original input — but it produces the identical output.
That is the nullspace at work: 3072 free dimensions of inputs all mapping to the same place.

In [18]:
weights = np_(layer.output.dense.weight)
bias = np_(layer.output.dense.bias)

for i in range(SEQ_LEN):
    our_output = weights @ resultingDenseInput[i] + bias
    original_output = np_(outputDenseOutput[0][i])

    is_close = np.allclose(our_output, original_output, rtol=1e-5, atol=1e-5)
    diff_magnitude = np.linalg.norm(our_output - original_output)

    print(f"\nToken {i}:")
    print(f"Outputs match within tolerance? {is_close}")
    print(f"Difference magnitude: {diff_magnitude}")

    original_input = np_(outputDenseInput[0][i])
    print(f"Distance between our input and the true input: "
          f"{np.linalg.norm(resultingDenseInput[i] - original_input):.4f}")


Token 0:
Outputs match within tolerance? True
Difference magnitude: 1.1020504851580498e-06
Distance between our input and the true input: 2.4666

Token 1:
Outputs match within tolerance? True
Difference magnitude: 1.742878372092577e-06
Distance between our input and the true input: 3.0813

Token 2:
Outputs match within tolerance? True
Difference magnitude: 1.484624240050361e-06
Distance between our input and the true input: 2.9578

Token 3:
Outputs match within tolerance? True
Difference magnitude: 2.030040367317817e-06
Distance between our input and the true input: 3.1103

Token 4:
Outputs match within tolerance? True
Difference magnitude: 2.5827635450680683e-06
Distance between our input and the true input: 3.8195


Now pick the `z` that recovers the *actual* input rather than the minimum-norm one.

**Slow cell** — a 4096x4096 pseudoinverse per token, roughly 5 minutes total.

In [19]:
for i in range(SEQ_LEN):
    target_output = outputDenseOutputNoBias[i]
    original_input = np_(outputDenseInput[0][i])

    print(f"\nToken {i}:")
    z = find_z_for_exact_match(original_input, target_output, weights)

    exactSolution = solutionGens[i](z)
    print('recovered:', exactSolution[:5], '...')
    print('actual:   ', original_input[:5], '...')
    print(f'max abs error: {np.abs(exactSolution - original_input).max():.3e}')


Token 0:


Reconstruction error: 1.19508327359428e-06


Nullspace component magnitude: 2.466559921991723
Particular solution magnitude: 1.3765213
recovered: [-1.74578152e-05 -5.15382998e-02 -6.60899291e-05 -8.48854358e-03
 -7.59451205e-03] ...
actual:    [-1.7453845e-05 -5.1538285e-02 -6.6087785e-05 -8.4885061e-03
 -7.5945212e-03] ...
max abs error: 1.292e-07

Token 1:


Reconstruction error: 1.6537761400482402e-06


Nullspace component magnitude: 3.081326469296225
Particular solution magnitude: 1.8995526
recovered: [-0.00018228 -0.05970772 -0.00033348 -0.03659311 -0.0105123 ] ...
actual:    [-0.00018225 -0.05970772 -0.00033347 -0.03659313 -0.01051229] ...
max abs error: 2.302e-07

Token 2:


...

Nullspace component magnitude: 2.9577648933785294
Particular solution magnitude: 1.7386352
recovered: [-5.35759402e-05 -6.57048531e-02 -2.54246201e-04 -3.63065980e-02
 -3.27821873e-03] ...
actual:    [-5.3587835e-05 -6.5704830e-02 -2.5426855e-04 -3.6306556e-02
 -3.2782168e-03] ...
max abs error: 1.541e-07

Token 3:


Reconstruction error: 1.305731246845965e-06


Nullspace component magnitude: 3.110316908183275
Particular solution magnitude: 1.9867663
recovered: [-0.00018565 -0.08795734 -0.0009199  -0.10897295 -0.00898421] ...
actual:    [-0.00018564 -0.08795736 -0.00091989 -0.10897295 -0.00898422] ...
max abs error: 1.275e-07

Token 4:


Reconstruction error: 1.8166934888145844e-06


Nullspace component magnitude: 3.819522738996925
Particular solution magnitude: 2.301452
recovered: [-1.18140288e-05 -1.11583326e-01 -1.82898018e-04 -3.17890180e-02
 -1.77414468e-02] ...
actual:    [-1.1813442e-05 -1.1158333e-01 -1.8290733e-04 -3.1789001e-02
 -1.7741427e-02] ...
max abs error: 1.659e-07


### 3.2 `output.LayerNorm` — only the learned affine comes back

In [20]:
with capture(layer.output.LayerNorm) as cap:
    layer(layer_input)
layerNormInput, layerNormOutput = cap['input'], cap['output']

print('LayerNorm Input:')
print(layerNormInput[0])
print('LayerNorm Output:')
print(layerNormOutput[0])

LayerNorm Input:
tensor([[ 0.5554,  0.4269,  0.9111,  ..., -0.3931,  0.7964,  1.2763],
        [ 0.5362, -0.4190,  0.8229,  ..., -0.2630,  0.9067,  1.4299],
        [ 1.0879, -0.0599,  1.1569,  ..., -0.6003,  0.8776,  1.0313],
        [ 1.2613, -0.0517,  1.3459,  ..., -0.4244,  1.4467,  1.2856],
        [ 0.9892, -0.0169,  1.3106,  ..., -0.0136,  1.1891,  1.0023]],
       grad_fn=<SelectBackward0>)
LayerNorm Output:
tensor([[ 0.3576,  0.3042,  0.5685,  ..., -0.2619,  0.3737,  0.6564],
        [ 0.3397, -0.2081,  0.5046,  ..., -0.1825,  0.4268,  0.7270],
        [ 0.6616,  0.0069,  0.7108,  ..., -0.3792,  0.4171,  0.5104],
        [ 0.7522,  0.0126,  0.8135,  ..., -0.2740,  0.7368,  0.6462],
        [ 0.6640,  0.0345,  0.8832,  ..., -0.0384,  0.6693,  0.5532]],
       grad_fn=<SelectBackward0>)


In [21]:
lnWeight = np_(layer.output.LayerNorm.weight)
lnBias = np_(layer.output.LayerNorm.bias)

generatedInput = []
for row in layerNormOutput[0]:
    result = generate_possible_inputs(np_(row), lnWeight, lnBias, 1, 1)
    generatedInput.append(result)

generatedInput = torch.tensor([generatedInput], dtype=torch.float32)
print('A candidate input we invented (scale=1, shift=1):')
print(generatedInput)

A candidate input we invented (scale=1, shift=1):
tensor([[[1.4364, 1.3310, 1.7280,  ..., 0.6589, 1.6339, 2.0273],
         [1.4112, 0.6452, 1.6410,  ..., 0.7704, 1.7083, 2.1277],
         [1.8654, 0.9331, 1.9215,  ..., 0.4941, 1.6946, 1.8195],
         [1.9933, 0.9407, 2.0611,  ..., 0.6419, 2.1419, 2.0128],
         [1.8688, 0.9700, 2.1559,  ..., 0.9729, 2.0474, 1.8805]]])


In [22]:
outputOfGeneratedInput = layer.output.LayerNorm(generatedInput)

print('Original LayerNorm output:')
print(layerNormOutput)
print('\nOutput of our invented input:')
print(outputOfGeneratedInput)
print(f'\nMax difference: {(outputOfGeneratedInput - layerNormOutput).abs().max().item():.3e}')
print(f'But our input differs from the real one by: '
      f'{(generatedInput - layerNormInput).abs().max().item():.3f}')

Original LayerNorm output:
tensor([[[ 0.3576,  0.3042,  0.5685,  ..., -0.2619,  0.3737,  0.6564],
         [ 0.3397, -0.2081,  0.5046,  ..., -0.1825,  0.4268,  0.7270],
         [ 0.6616,  0.0069,  0.7108,  ..., -0.3792,  0.4171,  0.5104],
         [ 0.7522,  0.0126,  0.8135,  ..., -0.2740,  0.7368,  0.6462],
         [ 0.6640,  0.0345,  0.8832,  ..., -0.0384,  0.6693,  0.5532]]],
       grad_fn=<NativeLayerNormBackward0>)

Output of our invented input:


tensor([[[ 0.3576,  0.3042,  0.5685,  ..., -0.2619,  0.3737,  0.6564],
         [ 0.3397, -0.2081,  0.5046,  ..., -0.1825,  0.4268,  0.7270],
         [ 0.6616,  0.0069,  0.7108,  ..., -0.3792,  0.4171,  0.5104],
         [ 0.7522,  0.0126,  0.8135,  ..., -0.2740,  0.7368,  0.6462],
         [ 0.6640,  0.0345,  0.8832,  ..., -0.0384,  0.6693,  0.5532]]],
       grad_fn=<NativeLayerNormBackward0>)

Max difference: 4.768e-07
But our input differs from the real one by: 4.814


Same output, different input — because scale=1 and shift=1 were arbitrary guesses.

But almost nothing was actually lost. Undoing the learned affine recovers the normalized vector
at **cosine 1.00000000** to the true centered input; supplying the input's real mean and std
reconstructs it to ~1e-06. LayerNorm destroys exactly 2 scalars per token, not the signal.

### 3.3 GELU

In [23]:
with capture(layer.intermediate.intermediate_act_fn) as cap:
    layer(layer_input)
geluInput, geluOutput = cap['input'], cap['output']

print('GELU input: ', geluInput.shape)
print('GELU output:', geluOutput.shape)

reversedGELU = inverse_gelu_matrix(np_(geluOutput[0]))

print('\nRe-applying GELU to our reversed values:')
print(gelu_vectorized(reversedGELU))
print('\nThe true GELU output:')
print(geluOutput[0])
print(f'\nMax error: {np.abs(gelu_vectorized(reversedGELU) - np_(geluOutput[0])).max():.3e}')
print(f'Max error vs the true pre-GELU input: '
      f'{np.abs(reversedGELU - np_(geluInput[0])).max():.3e}')

GELU input:  torch.Size([1, 5, 4096])
GELU output: torch.Size([1, 5, 4096])



Re-applying GELU to our reversed values:
[[-1.74538454e-05 -5.15382849e-02 -6.60877849e-05 ... -9.90133121e-05
  -1.74453110e-02 -1.43023673e-04]
 [-1.82252523e-04 -5.97077236e-02 -3.33468488e-04 ... -1.12267875e-03
  -2.04958729e-02 -7.14619382e-05]
 [-5.35878353e-05 -6.57048300e-02 -2.54268554e-04 ... -7.29811291e-05
  -2.71093026e-02 -1.15921910e-04]
 [-1.85635959e-04 -8.79573599e-02 -9.19893908e-04 ... -7.37973824e-05
  -4.25377190e-02 -5.44156173e-05]
 [-1.18134421e-05 -1.11583330e-01 -1.82907330e-04 ... -1.17000728e-03
  -4.14066315e-02 -1.93693719e-04]]

The true GELU output:
tensor([[-1.7454e-05, -5.1538e-02, -6.6088e-05,  ..., -9.9013e-05,
         -1.7445e-02, -1.4302e-04],
        [-1.8225e-04, -5.9708e-02, -3.3347e-04,  ..., -1.1227e-03,
         -2.0496e-02, -7.1462e-05],
        [-5.3588e-05, -6.5705e-02, -2.5427e-04,  ..., -7.2981e-05,
         -2.7109e-02, -1.1592e-04],
        [-1.8564e-04, -8.7957e-02, -9.1989e-04,  ..., -7.3797e-05,
         -4.2538e-02, -5.4416e-05

**This is worse than a two-branch problem.** Re-applying GELU to the reversed values matches
the true output to ~1e-05, so Newton found genuine preimages — but they differ from the true
pre-GELU input by as much as **16.4**.

The reason: GELU tends to 0 from below as x goes to negative infinity. So the narrow output band
[-0.17, 0) contains preimages ranging from -0.75 all the way down, and nothing in the output
distinguishes them. Positive outputs come back exactly; the entire negative tail is ambiguous.

### 3.4 `intermediate.dense` — the FFN projection up (1024 -> 4096)

In [24]:
with capture(layer.intermediate.dense) as cap:
    layer(layer_input)
intermediateDenseInput, intermediateDenseOutput = cap['input'], cap['output']

intermediateDenseNoBias = np_(intermediateDenseOutput[0] - layer.intermediate.dense.bias)
intermediateWeight = np_(layer.intermediate.dense.weight)
print(f'weight shape: {intermediateWeight.shape}  (4096 out x 1024 in)')
print(f'nullspace dimension: {checkNullspace(intermediateWeight)}')

solutionGens = []
for i in tqdm(intermediateDenseNoBias, desc='building solution families'):
    solutionGens.append(allInputsForMatrixProduct(i, intermediateWeight))

weight shape: (4096, 1024)  (4096 out x 1024 in)


nullspace dimension: 0


building solution families:   0%|          | 0/5 [00:00<?, ?it/s]

building solution families:  20%|██        | 1/5 [00:22<01:28, 22.15s/it]

building solution families:  40%|████      | 2/5 [00:28<00:38, 12.78s/it]

building solution families:  60%|██████    | 3/5 [00:41<00:25, 12.76s/it]

building solution families:  80%|████████  | 4/5 [00:46<00:09,  9.77s/it]

building solution families: 100%|██████████| 5/5 [01:04<00:00, 12.76s/it]

building solution families: 100%|██████████| 5/5 [01:04<00:00, 12.87s/it]

In [25]:
dims = np.zeros(HIDDEN_DIM)
resultingIntermediateDenseInput = [gen(dims) for gen in solutionGens]

print('\nOur reconstructed input:')
print(torch.tensor(np.array(resultingIntermediateDenseInput)))
print('\nThe actual input:')
print(intermediateDenseInput[0])
print(f'\nMax error: '
      f'{np.abs(np.array(resultingIntermediateDenseInput) - np_(intermediateDenseInput[0])).max():.3e}')

Nullspace component magnitude: 0.0
Particular solution magnitude: 36.94668
Nullspace component magnitude: 0.0
Particular solution magnitude: 36.90489
Nullspace component magnitude: 0.0
Particular solution magnitude: 36.39417
Nullspace component magnitude: 0.0
Particular solution magnitude: 36.05985
Nullspace component magnitude: 0.0
Particular solution magnitude: 29.906399

Our reconstructed input:
tensor([[ 0.5806,  0.5752,  0.9632,  ..., -0.3718,  0.8111,  1.2430],
        [ 0.6745, -0.3170,  1.0144,  ..., -0.2647,  0.9328,  1.5550],
        [ 1.1377,  0.0539,  1.1990,  ..., -0.5584,  0.9405,  0.9910],
        [ 1.3365,  0.0981,  1.4809,  ..., -0.3867,  1.4664,  1.2710],
        [ 1.0916,  0.1154,  1.4492,  ...,  0.1694,  1.1500,  1.0641]],
       dtype=torch.float64)

The actual input:
tensor([[ 0.5806,  0.5752,  0.9632,  ..., -0.3718,  0.8111,  1.2430],
        [ 0.6745, -0.3170,  1.0144,  ..., -0.2647,  0.9328,  1.5550],
        [ 1.1377,  0.0539,  1.1990,  ..., -0.5584,  0.9405, 

**This one inverts exactly.** The matrix is tall (4096x1024), so it has full column rank and
an empty nullspace — one output, exactly one input. Notice the nullspace magnitude printed
above is ~0 regardless of the `z` passed in.

### 3.5 `attention.output.dense` — square, also exact

In [26]:
with capture(layer.attention.output.dense) as cap:
    layer(layer_input)
attnOutDenseInput, attnOutDenseOutput = cap['input'], cap['output']

attnOutNoBias = np_(attnOutDenseOutput[0] - layer.attention.output.dense.bias)
attnOutWeight = np_(layer.attention.output.dense.weight)
print(f'weight shape: {attnOutWeight.shape}, nullspace dimension: {checkNullspace(attnOutWeight)}')

solutionGens = [allInputsForMatrixProduct(i, attnOutWeight)
                for i in tqdm(attnOutNoBias, desc='building solution families')]

dims = np.zeros(HIDDEN_DIM)
resultingAttnOutInput = np.array([gen(dims) for gen in solutionGens])

print('\nOur reconstructed input:')
print(torch.tensor(resultingAttnOutInput))
print('\nThe actual input:')
print(attnOutDenseInput[0])
print(f'\nMax error: {np.abs(resultingAttnOutInput - np_(attnOutDenseInput[0])).max():.3e}')

weight shape: (1024, 1024), nullspace dimension: 0


building solution families:   0%|          | 0/5 [00:00<?, ?it/s]

building solution families:  20%|██        | 1/5 [00:02<00:11,  2.89s/it]

building solution families:  40%|████      | 2/5 [00:05<00:08,  2.94s/it]

building solution families:  60%|██████    | 3/5 [00:13<00:10,  5.13s/it]

building solution families:  80%|████████  | 4/5 [00:27<00:08,  8.49s/it]

building solution families: 100%|██████████| 5/5 [00:29<00:00,  6.32s/it]

building solution families: 100%|██████████| 5/5 [00:29<00:00,  5.94s/it]

Nullspace component magnitude: 0.0
Particular solution magnitude: 13.163023
Nullspace component magnitude: 0.0
Particular solution magnitude: 15.310797
Nullspace component magnitude: 0.0
Particular solution magnitude: 13.009989
Nullspace component magnitude: 0.0
Particular solution magnitude: 14.957601
Nullspace component magnitude: 0.0
Particular solution magnitude: 12.227352

Our reconstructed input:
tensor([[-0.2757,  0.2522,  0.2764,  ...,  0.2808,  0.0787, -0.3979],
        [-0.2769,  0.3125,  0.2724,  ...,  0.2945,  0.1057, -0.3363],
        [-0.2640,  0.2448,  0.2652,  ...,  0.2856,  0.0821, -0.3875],
        [-0.2807,  0.2615,  0.2779,  ...,  0.2939,  0.0766, -0.3723],
        [-0.2652,  0.2245,  0.2714,  ...,  0.2825,  0.0847, -0.3913]],
       dtype=torch.float64)

The actual input:
tensor([[-0.2757,  0.2522,  0.2764,  ...,  0.2808,  0.0787, -0.3979],
        [-0.2768,  0.3125,  0.2725,  ...,  0.2945,  0.1057, -0.3365],
        [-0.2640,  0.2448,  0.2652,  ...,  0.2856,  0.08

### 3.6 Self-attention

`X` appears three times in `softmax(XWq (XWk)^T / sqrt(d)) @ (XWv)`, so there is no clean
algebraic inversion of the whole thing. But if the attention probabilities are treated as
given, the value path *is* linear and inverts.

First reproduce the forward pass by hand so we hold the probabilities and values explicitly.

In [27]:
import math

with capture(layer.attention.self) as cap:
    layer(layer_input)
attentionInput, attentionOutput = cap['input'], cap['output'][0]

selfAttn = layer.attention.self
keys = attentionInput @ selfAttn.key.weight.T + selfAttn.key.bias
queries = attentionInput @ selfAttn.query.weight.T + selfAttn.query.bias
values = attentionInput @ selfAttn.value.weight.T + selfAttn.value.bias

# Reshape K,Q,V into heads: [batch, num_heads, seq_len, head_dim]
keys = keys.view(1, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
queries = queries.view(1, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)
values = values.view(1, SEQ_LEN, NUM_HEADS, HEAD_DIM).transpose(1, 2)

attentionScores = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(HEAD_DIM)
attentionProbs = F.softmax(attentionScores, dim=-1)

calculatedOutput = torch.matmul(attentionProbs, values)
calculatedOutput = calculatedOutput.transpose(1, 2).contiguous().view(1, SEQ_LEN, HIDDEN_DIM)

print('Our hand-computed attention output:')
print(calculatedOutput)
print('\nThe real attention output:')
print(attentionOutput)
print(f'\nMax difference: {(calculatedOutput[0] - attentionOutput).abs().max().item():.3e}')

Our hand-computed attention output:
tensor([[[-0.2757,  0.2522,  0.2764,  ...,  0.2808,  0.0787, -0.3979],
         [-0.2768,  0.3125,  0.2725,  ...,  0.2945,  0.1057, -0.3365],
         [-0.2640,  0.2448,  0.2652,  ...,  0.2856,  0.0821, -0.3875],
         [-0.2808,  0.2613,  0.2777,  ...,  0.2939,  0.0767, -0.3722],
         [-0.2653,  0.2244,  0.2714,  ...,  0.2825,  0.0847, -0.3912]]],
       grad_fn=<ViewBackward0>)

The real attention output:
tensor([[[-0.2757,  0.2522,  0.2764,  ...,  0.2808,  0.0787, -0.3979],
         [-0.2768,  0.3125,  0.2725,  ...,  0.2945,  0.1057, -0.3365],
         [-0.2640,  0.2448,  0.2652,  ...,  0.2856,  0.0821, -0.3875],
         [-0.2808,  0.2613,  0.2777,  ...,  0.2939,  0.0767, -0.3722],
         [-0.2653,  0.2244,  0.2714,  ...,  0.2825,  0.0847, -0.3912]]],
       grad_fn=<ViewBackward0>)

Max difference: 4.768e-07


In [28]:
solvedInput = solve_for_attention_input(
    attentionProbs, selfAttn.value.weight, selfAttn.value.bias, calculatedOutput,
    num_heads=NUM_HEADS, head_dim=HEAD_DIM, seq_len=SEQ_LEN)

print('Solved-for attention input:')
print(solvedInput)
print('\nThe actual attention input:')
print(attentionInput)

error = verify_solution(solvedInput, attentionProbs, selfAttn.value.weight, selfAttn.value.bias,
                        calculatedOutput, num_heads=NUM_HEADS, head_dim=HEAD_DIM, seq_len=SEQ_LEN)
print(f'\nDoes our solved input reproduce the output? error = {error:.3e}')
print(f'Distance to the true input: {(solvedInput - attentionInput).norm().item():.4f}')

Solved-for attention input:
tensor([[[ 3.4142e-01,  5.4280e-01,  3.1161e-01,  ..., -2.0696e-01,
           9.5381e-02,  5.0951e-01],
         [ 4.5706e-01, -3.9769e-01,  5.4913e-01,  ..., -1.0224e-01,
           3.5436e-01,  9.5864e-01],
         [ 8.8443e-01,  6.2180e-04,  5.8676e-01,  ..., -4.2458e-01,
           2.2873e-01,  2.8764e-01],
         [ 1.3001e+00,  1.4495e-01,  9.7295e-01,  ..., -2.5702e-01,
           8.5753e-01,  5.6786e-01],
         [ 4.6718e-01,  2.1449e-02,  3.6413e-01,  ...,  2.3548e-01,
           3.4975e-02,  4.9930e-02]]], grad_fn=<UnsafeViewBackward0>)

The actual attention input:
tensor([[[ 3.4113e-01,  5.4243e-01,  3.1149e-01,  ..., -2.0684e-01,
           9.5085e-02,  5.0976e-01],
         [ 4.5751e-01, -3.9699e-01,  5.4930e-01,  ..., -1.0244e-01,
           3.5461e-01,  9.5827e-01],
         [ 8.8448e-01,  6.6389e-04,  5.8672e-01,  ..., -4.2450e-01,
           2.2866e-01,  2.8760e-01],
         [ 1.3001e+00,  1.4492e-01,  9.7293e-01,  ..., -2.5691e-01,
  

**The value path inverts essentially exactly** — distance to the true input 0.0153, cosine
similarity 1.0000002. The residual is numerical conditioning (the 5x5 probability matrices have
condition numbers of 1e2-1e3), not genuine ambiguity.

Note the conditional: this required knowing `attentionProbs`. Given the attention pattern, the
rest of attention is solvable linear algebra. The pattern itself is what does not invert, since
the scores are quadratic in `X`. That reframes the problem usefully — the target is recovering
the attention pattern, not the input.

In [29]:
probs_np = np_(attentionProbs)[0]   # [num_heads, seq_len, seq_len]
head0 = probs_np[0]

solutions = get_softmax_input_space(head0, num_samples=5)

if verify_solutions(solutions, head0):
    print('All solutions verified: every one produces the same attention probabilities.\n')
    for i, sol in enumerate(solutions):
        print(f'Solution {i + 1} (row 0): {sol[0]}')
else:
    print('Verification failed')

print('\nThe true scores for head 0, row 0:')
print(np_(attentionScores)[0][0][0])

All solutions verified: every one produces the same attention probabilities.

Solution 1 (row 0): [-12.07779288 -11.49939954 -12.50497246 -12.17257237 -10.78533411]
Solution 2 (row 0): [-7.07779288 -6.49939954 -7.50497246 -7.17257237 -5.78533411]
Solution 3 (row 0): [-2.07779288 -1.49939954 -2.50497246 -2.17257237 -0.78533411]
Solution 4 (row 0): [2.92220712 3.50060046 2.49502754 2.82742763 4.21466589]
Solution 5 (row 0): [7.92220712 8.50060046 7.49502754 7.82742763 9.21466589]

The true scores for head 0, row 0:
[-1.2269442  -0.64855087 -1.6541238  -1.3217236   0.06551453]


### 3.7 The pooler — `tanh(Wx + b)`

In [30]:
W = np_(model.pooler.dense.weight)
b = np_(model.pooler.dense.bias)
x_original = np_(outputs.hidden_states[NUM_LAYERS][0][0])   # the [CLS] hidden state
y = np_(outputs.pooler_output[0])                           # the sentence embedding

print(f'nullspace dimension of the pooler weight: {checkNullspace(W)}')

solution_gen = find_all_possible_inputs(y, W, b)

z = np.random.randn(HIDDEN_DIM)
x_solution = solution_gen(z)

y_reconstructed = np.tanh(W @ x_solution + b)

print("\nOriginal y:", y[:5], '...')
print("Reconstructed y:", y_reconstructed[:5], '...')
print("\nOriginal x:", x_original[:5], '...')
print("Reconstructed x:", x_solution[:5], '...')
print("\nInput cosine similarity:", cosine_similarity(
    torch.tensor(x_original, dtype=torch.float32), torch.tensor(x_solution, dtype=torch.float32)))
print("Output cosine similarity:", cosine_similarity(
    torch.tensor(y, dtype=torch.float32), torch.tensor(y_reconstructed, dtype=torch.float32)))

nullspace dimension of the pooler weight: 0


Nullspace component magnitude: 0.0013863684418070712
Particular solution magnitude: 16.759146

Original y: [-0.94499004 -0.8097364  -0.83618593 -0.8973111  -0.5444824 ] ...
Reconstructed y: [-0.94499932 -0.80974777 -0.83618284 -0.89731222 -0.54449135] ...

Original x: [0.3575791  0.3042189  0.56853694 0.29111907 0.5117589 ] ...
Reconstructed x: [0.3577588  0.30412507 0.5685056  0.29098609 0.51173093] ...

Input cosine similarity: 0.9999998807907104
Output cosine similarity: 1.0


## 4. Which weight matrices are invertible at all?

Run both tolerances side by side. The strict one says every projection is full rank; the
honest one says otherwise.

In [31]:
print(f'{"layer":>5} | {"strict tol=1e-10":>18} | {"float32 tol":>14} | {"cond (K/Q/V)":>20}')
strict_total = proper_total = 0
for i in range(NUM_LAYERS):
    attn = model.encoder.layer[i].attention.self
    W = [np_(attn.key.weight), np_(attn.query.weight), np_(attn.value.weight)]
    strict = [int(checkNullspace(w)) for w in W]
    proper = [numericalRank(w) for w in W]
    strict_total += sum(strict); proper_total += sum(n for n, _ in proper)
    print(f'{i:>5} | {str(strict):>18} | {str([n for n, _ in proper]):>14} | '
          f'{" ".join(f"{c:.0e}" for _, c in proper):>20}')

print(f'\nDimensions declared lost across all 72 matrices:')
print(f'  strict tol=1e-10 : {strict_total}   <- what the notebook originally reported')
print(f'  float32 tolerance: {proper_total}   <- what the arithmetic actually supports')

layer |   strict tol=1e-10 |    float32 tol |         cond (K/Q/V)


    0 |          [0, 0, 0] |      [2, 1, 1] |    2e+04 1e+04 3e+05


    1 |          [0, 0, 0] |      [1, 2, 0] |    6e+04 4e+04 7e+03


    2 |          [0, 0, 0] |      [1, 1, 0] |    1e+04 9e+03 3e+03


    3 |          [0, 0, 0] |      [1, 0, 1] |    8e+05 8e+03 8e+03


...

   19 |          [0, 0, 0] |      [0, 0, 0] |    6e+03 8e+03 4e+03


   20 |          [0, 0, 0] |      [0, 0, 1] |    7e+03 4e+03 1e+04


   21 |          [0, 0, 0] |      [1, 1, 1] |    1e+04 1e+04 1e+04


   22 |          [0, 0, 0] |      [1, 2, 0] |    9e+03 5e+05 3e+03


   23 |          [0, 0, 0] |      [0, 0, 0] |    7e+03 5e+03 4e+03

Dimensions declared lost across all 72 matrices:
  strict tol=1e-10 : 0   <- what the notebook originally reported
  float32 tolerance: 51   <- what the arithmetic actually supports


## 5. Structural vs. contingent reversibility

A crucial distinction. Some pieces reverse because of **structure** — the shape of the
operation guarantees it, for any weights. Others reverse only because of a **contingent
property of these particular trained weights**, and would stop working if the weights
were different, the input were longer, or the arithmetic were less precise.

Everything below measures a contingent condition, and says what would break it.

### 5.1 Conditioning — the difference between "invertible" and "usefully invertible"

Full rank is binary; conditioning is the question that actually matters. Inverting a matrix
amplifies any error in the input by roughly its condition number.

In [32]:
hdr = f'{"matrix":<22}{"shape":<14}{"sigma_min":>11}{"cond":>10}{"dims lost":>18}{"digits left":>13}'
print(hdr); print('-' * len(hdr))
specs = [('query', lambda l: l.attention.self.query.weight),
         ('key', lambda l: l.attention.self.key.weight),
         ('value', lambda l: l.attention.self.value.weight),
         ('attn.output.dense', lambda l: l.attention.output.dense.weight),
         ('intermediate.dense', lambda l: l.intermediate.dense.weight),
         ('output.dense', lambda l: l.output.dense.weight)]

for name, get in specs:
    conds, lost = [], []
    for i in range(NUM_LAYERS):
        n, c = numericalRank(np_(get(model.encoder.layer[i])))
        conds.append(c); lost.append(n)
    W = np_(get(model.encoder.layer[0]))
    s = np.linalg.svd(W, compute_uv=False)
    med = float(np.median(conds))
    print(f'{name:<22}{str(W.shape):<14}{s[min(W.shape)-1]:>11.1e}{med:>10.1e}'
          f'{f"{sum(lost)} over 24 layers":>18}{max(0, 7 - np.log10(med)):>13.1f}')

n, c = numericalRank(np_(model.pooler.dense.weight))
sp = np.linalg.svd(np_(model.pooler.dense.weight), compute_uv=False)
print(f'{"pooler.dense":<22}{"(1024, 1024)":<14}{sp[-1]:>11.1e}{c:>10.1e}'
      f'{f"{n} (single matrix)":>18}{max(0, 7 - np.log10(c)):>13.1f}')
print('\n"digits left" = how many of float32\'s ~7 significant digits survive one inversion.')

matrix                shape           sigma_min      cond         dims lost  digits left
----------------------------------------------------------------------------------------


query                 (1024, 1024)      4.6e-04   9.7e+03 17 over 24 layers          3.0


key                   (1024, 1024)      2.8e-04   1.2e+04 20 over 24 layers          2.9


value                 (1024, 1024)      8.1e-06   8.7e+03 14 over 24 layers          3.1


attn.output.dense     (1024, 1024)      9.4e-04   7.7e+03 10 over 24 layers          3.1


intermediate.dense    (4096, 1024)      6.1e-01   3.4e+01  0 over 24 layers          5.5


output.dense          (1024, 4096)      6.5e-01   1.8e+01  0 over 24 layers          5.7


pooler.dense          (1024, 1024)      2.7e-04   6.3e+04 3 (single matrix)          2.2

"digits left" = how many of float32's ~7 significant digits survive one inversion.


**Careful with the "dims lost" column — it is a threshold crossing, not a grade.**

A matrix is flagged when its smallest singular value dips under the tolerance line. That makes
it tempting to read "15 of 24 layers flagged" as "the other 9 are fine." They are not. The
flag is a cliff edge drawn through a continuous population, and the layers on either side of
it are neighbours. The next two cells show the spread.

In [33]:
print('QUERY projection, every layer.  "flagged" = smallest singular value fell below the tolerance\n')
print(f'{"layer":>5}{"sigma_min":>12}{"rank tol":>12}{"cond":>10}{"digits left":>13}{"flagged":>9}')
rows = []
for i in range(NUM_LAYERS):
    W = np_(model.encoder.layer[i].attention.self.query.weight)
    s = np.linalg.svd(W, compute_uv=False)
    tol = s[0] * max(W.shape) * EPS32
    c = s[0] / s[min(W.shape) - 1]
    flagged = int((s < tol).sum()) > 0
    rows.append((c, flagged))
    print(f'{i:>5}{s[min(W.shape)-1]:>12.2e}{tol:>12.2e}{c:>10.1e}'
          f'{max(0, 7-np.log10(c)):>13.1f}{("YES" if flagged else "-"):>9}')

flag = [c for c, f in rows if f]
clean = [c for c, f in rows if not f]
print(f'\nflagged   (n={len(flag)}): cond {min(flag):.1e} .. {max(flag):.1e}')
print(f'unflagged (n={len(clean)}): cond {min(clean):.1e} .. {max(clean):.1e}')
print('\nThe two ranges are adjacent, not separated. Being unflagged buys you almost nothing.')

QUERY projection, every layer.  "flagged" = smallest singular value fell below the tolerance

layer   sigma_min    rank tol      cond  digits left  flagged


    0    4.59e-04    6.57e-04   1.2e+04          2.9      YES


    1    1.34e-04    6.47e-04   4.0e+04          2.4      YES


    2    6.10e-04    6.54e-04   8.8e+03          3.1      YES


    3    7.36e-04    6.99e-04   7.8e+03          3.1        -


...

   19    7.62e-04    7.16e-04   7.7e+03          3.1        -


   20    1.34e-03    7.17e-04   4.4e+03          3.4        -


   21    5.13e-04    6.62e-04   1.1e+04          3.0      YES


   22    1.38e-05    8.09e-04   4.8e+05          1.3      YES


   23    1.51e-03    8.90e-04   4.8e+03          3.3        -

flagged   (n=15): cond 8.5e+03 .. 5.0e+05
unflagged (n=9): cond 4.4e+03 .. 7.8e+03

The two ranges are adjacent, not separated. Being unflagged buys you almost nothing.


In [34]:
allc = []
for i in range(NUM_LAYERS):
    a = model.encoder.layer[i].attention.self
    for W in (a.query.weight, a.key.weight, a.value.weight):
        allc.append(numericalRank(np_(W))[1])
allc = np.array(allc)

print('All 72 K/Q/V matrices, bucketed by condition number:')
for thr, label in [(1e2, 'cond < 1e2  (safe,  ~5 digits survive)'),
                   (1e3, 'cond < 1e3  (ok,    ~4 digits survive)'),
                   (1e4, 'cond < 1e4  (       ~3 digits survive)'),
                   (1e5, 'cond < 1e5  (       ~2 digits survive)')]:
    print(f'  {label:<40} {int((allc < thr).sum()):>3} / 72')
print(f'\n  best of all 72: {allc.min():.1e}   median: {np.median(allc):.1e}')

ffn = []
for i in range(NUM_LAYERS):
    l = model.encoder.layer[i]
    ffn += [numericalRank(np_(l.intermediate.dense.weight))[1],
            numericalRank(np_(l.output.dense.weight))[1]]
ffn = np.array(ffn)
print(f'\nFor contrast, the 48 FFN matrices: cond {ffn.min():.1f} .. {ffn.max():.1f} '
      f'(median {np.median(ffn):.1f}), none flagged.')
print('Three orders of magnitude better. That is what a genuinely reversible matrix looks like.')

All 72 K/Q/V matrices, bucketed by condition number:
  cond < 1e2  (safe,  ~5 digits survive)     0 / 72
  cond < 1e3  (ok,    ~4 digits survive)     0 / 72
  cond < 1e4  (       ~3 digits survive)    33 / 72
  cond < 1e5  (       ~2 digits survive)    63 / 72

  best of all 72: 2.6e+03   median: 1.1e+04



For contrast, the 48 FFN matrices: cond 13.8 .. 74.2 (median 31.1), none flagged.
Three orders of magnitude better. That is what a genuinely reversible matrix looks like.


**Not one of the 72 attention projections is well conditioned.** The best sits at 2.6e3 — every
single one burns at least 3 of float32's 7 digits, flagged or not. Three layers (10, 16, 22) are
catastrophic at ~5e5, leaving barely one digit.

So the answer to "are the unflagged layers reversible?" is no. There is no clean subset of
attention layers that inverts safely; there is one poorly-conditioned population, some of which
happens to fall on the far side of an arbitrary line. The FFN matrices, sitting three orders of
magnitude lower, are the only thing here that inverts robustly.

### 5.2 The pooler's tanh: how close to saturation?

`arctanh` is exact for |y| < 1 and infinite at |y| = 1. Contingent on the activations never
reaching the rail.

In [35]:
y = np_(outputs.pooler_output[0])
mx = np.abs(y).max()
print(f'max |pooler_output| = {mx:.8f}   margin to 1.0 = {1 - mx:.3e}')
print(f'components above 0.999: {(np.abs(y) > 0.999).sum()} / {y.size}')
print(f'error amplification of arctanh at that point: {1 / (1 - mx**2):.0f}x')
print('\nSafe here -- but 8 components sit where arctanh magnifies error ~9000x.')

max |pooler_output| = 0.99994504   margin to 1.0 = 5.496e-05
components above 0.999: 8 / 1024
error amplification of arctanh at that point: 9098x

Safe here -- but 8 components sit where arctanh magnifies error ~9000x.


### 5.3 LayerNorm: is dividing by gamma safe?

Undoing the learned affine divides by `weight`. A near-zero component would blow up.

In [36]:
gammas = [model.embeddings.LayerNorm.weight.abs().min().item()]
for i in range(NUM_LAYERS):
    l = model.encoder.layer[i]
    gammas.append(l.attention.output.LayerNorm.weight.abs().min().item())
    gammas.append(l.output.LayerNorm.weight.abs().min().item())
print(f'smallest |gamma| across all {len(gammas)} LayerNorms: {min(gammas):.4f}')
print('Comfortably far from zero -- this contingent condition holds easily.')

smallest |gamma| across all 49 LayerNorms: 0.0478
Comfortably far from zero -- this contingent condition holds easily.


### 5.4 GELU: how much of the signal is actually on the ambiguous branch?

GELU is non-injective below x = -0.7517. That is a structural fact. Whether it *matters*
depends entirely on where the real activations sit.

In [37]:
store = {}
def mk(i):
    def hook_function(m, inp, o):
        store[i] = inp[0]
    return hook_function

handles = [model.encoder.layer[i].intermediate.intermediate_act_fn.register_forward_hook(mk(i))
           for i in range(NUM_LAYERS)]
model(**encoded)
for h in handles:
    h.remove()

frac = [100 * (store[i] < -0.7517).float().mean().item() for i in range(NUM_LAYERS)]
total = np.mean(frac)
print(f'pre-GELU activations on the ambiguous branch: {total:.1f}% overall')
print(f'per layer: min {min(frac):.1f}% (layer {frac.index(min(frac))}), '
      f'max {max(frac):.1f}% (layer {frac.index(max(frac))})')
print('\nThis is not an edge case. Nearly everything is ambiguous -- which is why the')
print('reversed GELU was off by 16.4 despite reproducing the output to 1e-05.')

pre-GELU activations on the ambiguous branch: 89.7% overall
per layer: min 76.4% (layer 18), max 98.9% (layer 23)

This is not an edge case. Nearly everything is ambiguous -- which is why the
reversed GELU was off by 16.4 despite reproducing the output to 1e-05.


### 5.5 Attention probabilities: invertible, but only for a short phrase

Recovering the value path needs the SxS probability matrix to invert. Softmax rows are
positive and sum to 1; nothing guarantees independence. Peaked attention or two tokens
attending identically would make it singular.

In [38]:
attnOut = model(**encoded, output_attentions=True)
conds = []
for i in range(NUM_LAYERS):
    A = attnOut.attentions[i][0].double()
    for h in range(A.shape[0]):
        conds.append(torch.linalg.cond(A[h]).item())
conds = np.array(conds)

print(f'{len(conds)} matrices ({NUM_LAYERS} layers x {NUM_HEADS} heads), size {SEQ_LEN}x{SEQ_LEN}')
print(f'condition number -- median {np.median(conds):.1e}, best {conds.min():.1e}, worst {conds.max():.1e}')
print(f'effectively singular in float32 (cond > 1e6): {100 * (conds > 1e6).mean():.1f}%')
print(f'already shaky (cond > 1e4):                   {100 * (conds > 1e4).mean():.1f}%')
print('\nHolds for 5 tokens. Longer sequences give more chances for near-dependence.')

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


384 matrices (24 layers x 16 heads), size 5x5
condition number -- median 5.0e+02, best 1.2e+01, worst 3.7e+05
effectively singular in float32 (cond > 1e6): 0.0%
already shaky (cond > 1e4):                   1.6%

Holds for 5 tokens. Longer sequences give more chances for near-dependence.


### 5.6 The embedding table: is the token lookup unique?

Reversing the very first step means identifying which vocabulary row you are holding.
Contingent on no two rows being identical -- and on them being far enough apart to survive noise.

In [39]:
E = model.embeddings.word_embeddings.weight
seen, dup = set(), 0
for row in E:
    k = row.detach().numpy().tobytes()
    if k in seen:
        dup += 1
    else:
        seen.add(k)
print(f'table {tuple(E.shape)} -- exact duplicate rows: {dup}')

idx = torch.randperm(E.shape[0])[:300]
d = torch.cdist(E[idx], E)
d[torch.arange(300), idx] = float('inf')
nn = d.min(dim=1).values
print(f'nearest-neighbour distance (300 sampled tokens): min {nn.min():.4f}, median {nn.median():.4f}')
print(f'median row norm: {E.norm(dim=1).median():.4f} '
      f'-> closest pair is {100 * nn.min() / E.norm(dim=1).median():.0f}% of a row norm apart')
print('\nRobust: no collisions, and generous separation.')

table (30522, 1024) -- exact duplicate rows: 0


nearest-neighbour distance (300 sampled tokens): min 0.3444, median 1.2780


median row norm: 1.4644 -> closest pair is 24% of a row norm apart

Robust: no collisions, and generous separation.


## 6. Where this stands

Separating what is guaranteed from what merely happens to hold for these weights:

### Reversible by structure — true for any weights

| Piece | Why |
|---|---|
| Tokenizer | Dictionary lookup both ways |
| Softmax | Loses exactly 1 constant per row, always |
| LayerNorm | Loses exactly 2 scalars per token (mean, std), always — direction fully preserved |

### Reversible only because of these weights — and robustly so

| Piece | The contingent condition | Margin |
|---|---|---|
| `intermediate.dense` (1024 -> 4096) | Full column rank *and* well conditioned | cond 34, 0/24 layers deficient — measured err 9.5e-06 |
| Embedding lookup | No duplicate vocabulary rows | 0 collisions, nearest pair 24% of a row norm |
| LayerNorm affine | No near-zero gamma | smallest |gamma| = 0.048 |

### Reversible here, but fragile — would break easily

| Piece | The contingent condition | Why it is fragile |
|---|---|---|
| Q / K / V projections | Full rank | Only at tol=1e-10. **0 of 72 have cond < 1e3**; best 2.6e3, median 1.1e4, worst 7.6e5. The flagged/unflagged split is a threshold artifact — no subset is safe |
| `attention.output.dense` | Full rank | median cond 7.7e3 — ~3 of 7 digits survive, in every layer |
| Pooler `tanh(Wx+b)` | No saturation, invertible W | Margin to |y|=1 is 5.5e-05; cond 6.3e4 |
| Attention value path | SxS probs invertible | Holds at 5 tokens (0% singular); peaked or longer attention would not |

### Not reversible

| Piece | Why |
|---|---|
| GELU | Non-injective below -0.7517 — and **89.7% of real activations are there** (98.9% in layer 23) |
| `output.dense` (4096 -> 1024) | 3072-dim nullspace by shape. *But* it is well conditioned (cond 18) on what it does keep |
| Attention scores | Quadratic in `X` — no pseudoinverse exists |
| Residual connections | `y = x + f(x)`, same unknown both sides. Never attempted here |

### The honest summary

Only **one** piece of this network is unconditionally, robustly invertible: the FFN's
up-projection. Everything else is either structurally lossy, or works solely because these
particular weights happen to cooperate — and the attention projections barely cooperate,
clinging to full rank by a margin thinner than float32 can represent — and that is true of every one of them, not just the ones a threshold happens to flag.

The earlier claim that "all 72 Q/K/V matrices are full rank" was an artifact of a tolerance
set 1000x tighter than the arithmetic warrants.

**Most promising next step (untested):** `output.dense` discards 3072 of 4096 dimensions, but
its input came from a 1024-dim bottleneck two steps earlier, so the true answer lies on a
1024-dim surface inside the solution family. Intersecting the two should generically pin it
down — and since both FFN matrices are the well-conditioned ones, that intersection is the
part of the network where a numerical attack has the best chance of working.